In [1]:
# ================================================================
# AI SHADOW ANALYST
# AUTONOMOUS HIDDEN-PATTERN DISCOVERY ENGINE
# COMPLETE PROJECT — ONE CELL
# ================================================================

import sys
import subprocess
import importlib.util
import os
import time

# ================================================================
# INSTALL REQUIRED LIBRARIES
# ================================================================

packages = {
    "streamlit": "streamlit",
    "pandas": "pandas",
    "numpy": "numpy",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "plotly": "plotly",
}

for module, package in packages.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package
        ])

# ================================================================
# CREATE STREAMLIT APPLICATION
# ================================================================

app_code = r'''
import streamlit as st
import pandas as pd
import numpy as np

from scipy.stats import pearsonr, spearmanr, ttest_ind
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import silhouette_score

import plotly.express as px
import plotly.graph_objects as go


# ================================================================
# PAGE CONFIG
# ================================================================

st.set_page_config(
    page_title="AI Shadow Analyst",
    page_icon="🔎",
    layout="wide",
    initial_sidebar_state="expanded"
)


# ================================================================
# CUSTOM CSS
# ================================================================

st.markdown("""
<style>

[data-testid="stAppViewContainer"] {
    background: #f5f7fb;
}

.hero {
    padding: 35px;
    border-radius: 24px;
    margin-bottom: 25px;
    background: linear-gradient(
        135deg,
        #111827,
        #312e81,
        #1e1b4b
    );
    color: white;
}

.hero h1 {
    font-size: 45px;
    font-weight: 800;
    margin-bottom: 8px;
}

.hero p {
    font-size: 18px;
    color: #dbeafe;
}

.card {
    background: white;
    padding: 22px;
    border-radius: 18px;
    border: 1px solid #e5e7eb;
    margin-bottom: 16px;
}

.case-title {
    font-size: 21px;
    font-weight: 700;
}

.warning {
    background: #fffbeb;
    border-left: 5px solid #f59e0b;
    padding: 18px;
    border-radius: 10px;
}

.success {
    background: #ecfdf5;
    border-left: 5px solid #10b981;
    padding: 18px;
    border-radius: 10px;
}

.metric-card {
    background: white;
    padding: 18px;
    border-radius: 16px;
    text-align: center;
    border: 1px solid #e5e7eb;
}

</style>
""", unsafe_allow_html=True)


# ================================================================
# SYNTHETIC DATA GENERATOR
# ================================================================

@st.cache_data
def generate_data(n=6000, seed=42):

    rng = np.random.default_rng(seed)

    timestamps = (
        pd.Timestamp("2026-01-01")
        + pd.to_timedelta(
            rng.integers(
                0,
                180 * 24 * 60,
                n
            ),
            unit="m"
        )
    )

    user_id = rng.integers(
        10000,
        10350,
        n
    )

    action = rng.choice(
        [
            "Search",
            "View",
            "Edit",
            "Upload",
            "Download",
            "Share",
            "Review"
        ],
        n,
        p=[
            .18,
            .22,
            .15,
            .10,
            .10,
            .10,
            .15
        ]
    )

    environment = rng.choice(
        [
            "Web",
            "Mobile",
            "Desktop"
        ],
        n,
        p=[
            .50,
            .30,
            .20
        ]
    )

    duration = np.maximum(
        5,
        rng.normal(65, 22, n)
    )

    clicks = np.maximum(
        1,
        rng.poisson(8, n)
    )

    latency = np.maximum(
        20,
        rng.normal(175, 45, n)
    )

    errors = rng.poisson(
        .8,
        n
    )

    sessions = (
        rng.poisson(3, n)
        + 1
    )

    complexity = np.clip(
        rng.normal(50, 15, n),
        1,
        100
    )

    # ------------------------------------------------------------
    # BASE SUCCESS PROBABILITY
    # ------------------------------------------------------------

    probability = (
        .58
        + .08 * (duration > 65)
        - .11 * (errors >= 2)
        - .09 * (latency > 220)
        + .04 * (clicks > 10)
    )

    probability = np.clip(
        probability,
        .05,
        .95
    )

    success = (
        rng.random(n)
        <
        probability
    ).astype(int)

    # ------------------------------------------------------------
    # HIDDEN PATTERN 1
    # DURATION SWEET SPOT
    # ------------------------------------------------------------

    sweet_spot = (
        (duration >= 50)
        &
        (duration <= 85)
    )

    success = np.where(
        sweet_spot,
        (
            rng.random(n)
            <
            np.clip(
                probability + .13,
                .05,
                .97
            )
        ).astype(int),
        success
    )

    # ------------------------------------------------------------
    # HIDDEN PATTERN 2
    # MOBILE + HIGH LATENCY
    # ------------------------------------------------------------

    mobile_problem = (
        (environment == "Mobile")
        &
        (latency > 220)
    )

    success = np.where(
        mobile_problem,
        (
            rng.random(n) < .30
        ).astype(int),
        success
    )

    # ------------------------------------------------------------
    # HIDDEN PATTERN 3
    # FRIDAY EFFECT
    # ------------------------------------------------------------

    friday = (
        timestamps.dayofweek == 4
    )

    friday_problem = (
        friday
        &
        (errors >= 2)
    )

    success = np.where(
        friday_problem,
        (
            rng.random(n) < .28
        ).astype(int),
        success
    )

    # ------------------------------------------------------------
    # HIDDEN PATTERN 4
    # FAST SEARCH BEHAVIOR
    # ------------------------------------------------------------

    search_fast = (
        (action == "Search")
        &
        (duration < 55)
    )

    success = np.where(
        search_fast,
        (
            rng.random(n) < .78
        ).astype(int),
        success
    )

    # ------------------------------------------------------------
    # TEMPORAL DRIFT
    # ------------------------------------------------------------

    drift = (
        timestamps
        >=
        pd.Timestamp("2026-05-20")
    )

    latency = np.where(
        drift,
        latency * 1.15,
        latency
    )

    errors = np.where(
        drift,
        errors + rng.binomial(
            1,
            .12,
            n
        ),
        errors
    )

    # ------------------------------------------------------------
    # DERIVED FEATURES
    # ------------------------------------------------------------

    efficiency = (
        duration
        /
        np.maximum(
            clicks,
            1
        )
    )

    engagement = (
        clicks * .7
        +
        sessions * 1.4
        -
        errors * 2
    )

    satisfaction = (
        86
        -
        errors * 7
        -
        latency / 80
        +
        rng.normal(
            0,
            7,
            n
        )
    )

    satisfaction = np.clip(
        satisfaction,
        1,
        100
    )

    # ------------------------------------------------------------
    # ANOMALY POPULATION
    # ------------------------------------------------------------

    anomaly = (
        rng.random(n)
        <
        .025
    )

    duration[anomaly] *= rng.uniform(
        3,
        6,
        anomaly.sum()
    )

    clicks[anomaly] *= rng.integers(
        3,
        7,
        anomaly.sum()
    )

    latency[anomaly] *= rng.uniform(
        2,
        5,
        anomaly.sum()
    )

    errors[anomaly] += rng.integers(
        5,
        15,
        anomaly.sum()
    )

    satisfaction[anomaly] = np.clip(
        satisfaction[anomaly]
        -
        rng.uniform(
            20,
            50,
            anomaly.sum()
        ),
        1,
        100
    )

    # ------------------------------------------------------------
    # DATAFRAME
    # ------------------------------------------------------------

    df = pd.DataFrame({

        "timestamp": timestamps,

        "user_id": user_id,

        "action": action,

        "environment": environment,

        "duration_sec":
            np.round(
                duration,
                2
            ),

        "clicks":
            clicks.astype(int),

        "errors":
            errors.astype(int),

        "latency_ms":
            np.round(
                latency,
                2
            ),

        "sessions":
            sessions.astype(int),

        "complexity":
            np.round(
                complexity,
                2
            ),

        "efficiency":
            np.round(
                efficiency,
                3
            ),

        "engagement":
            np.round(
                engagement,
                2
            ),

        "satisfaction":
            np.round(
                satisfaction,
                2
            ),

        "success":
            success.astype(int)
    })

    # ------------------------------------------------------------
    # MISSING VALUES
    # ------------------------------------------------------------

    for column in [
        "duration_sec",
        "latency_ms",
        "satisfaction"
    ]:

        indexes = rng.choice(
            len(df),
            int(
                len(df) * .015
            ),
            replace=False
        )

        df.loc[
            indexes,
            column
        ] = np.nan

    return df


# ================================================================
# DATASET PROFILE
# ================================================================

def dataset_profile(df):

    numeric = df.select_dtypes(
        include=np.number
    )

    missing = int(
        df.isna().sum().sum()
    )

    duplicates = int(
        df.duplicated().sum()
    )

    total_cells = (
        len(df)
        *
        len(df.columns)
    )

    missing_pct = (
        missing
        /
        max(
            total_cells,
            1
        )
        *
        100
    )

    duplicate_pct = (
        duplicates
        /
        max(
            len(df),
            1
        )
        *
        100
    )

    return {

        "rows": len(df),

        "columns": len(
            df.columns
        ),

        "numeric": len(
            numeric.columns
        ),

        "missing": missing,

        "duplicates": duplicates,

        "missing_pct": missing_pct,

        "duplicate_pct": duplicate_pct
    }


# ================================================================
# RELATIONSHIP MINING
# ================================================================

def relationship_mining(df):

    numeric = df.select_dtypes(
        include=np.number
    ).columns.tolist()

    results = []

    for i in range(
        len(numeric)
    ):

        for j in range(
            i + 1,
            len(numeric)
        ):

            a = numeric[i]

            b = numeric[j]

            pair = df[
                [a, b]
            ].dropna()

            if len(pair) < 100:
                continue

            if (
                pair[a].nunique() < 5
                or
                pair[b].nunique() < 5
            ):
                continue

            try:

                pearson, p_value = pearsonr(
                    pair[a],
                    pair[b]
                )

                spearman, _ = spearmanr(
                    pair[a],
                    pair[b]
                )

                strength = (
                    abs(pearson) * .55
                    +
                    abs(spearman) * .45
                )

                significance = min(
                    1,
                    -np.log10(
                        max(
                            p_value,
                            1e-300
                        )
                    ) / 12
                )

                score = min(
                    100,
                    strength * 75
                    +
                    significance * 25
                )

                if score >= 30:

                    results.append({

                        "Feature A": a,

                        "Feature B": b,

                        "Pearson":
                            round(
                                pearson,
                                3
                            ),

                        "Spearman":
                            round(
                                spearman,
                                3
                            ),

                        "P-Value":
                            p_value,

                        "Interestingness":
                            round(
                                score,
                                2
                            )
                    })

            except Exception:
                pass

    if not results:

        return pd.DataFrame()

    return (
        pd.DataFrame(results)
        .sort_values(
            "Interestingness",
            ascending=False
        )
        .reset_index(drop=True)
    )


# ================================================================
# GROUP DIFFERENCE MINING
# ================================================================

def group_difference_mining(df):

    categorical = df.select_dtypes(
        exclude=np.number
    ).columns.tolist()

    numeric = df.select_dtypes(
        include=np.number
    ).columns.tolist()

    results = []

    for category in categorical:

        if category == "timestamp":
            continue

        if df[category].nunique() > 15:
            continue

        for metric in numeric:

            groups = []

            for value, group in df.groupby(
                category
            ):

                values = group[
                    metric
                ].dropna()

                if len(values) >= 40:

                    groups.append(
                        (
                            str(value),
                            values
                        )
                    )

            if len(groups) < 2:
                continue

            highest = max(
                groups,
                key=lambda x: x[1].mean()
            )

            lowest = min(
                groups,
                key=lambda x: x[1].mean()
            )

            try:

                statistic, p_value = ttest_ind(
                    highest[1],
                    lowest[1],
                    equal_var=False
                )

                pooled_std = np.sqrt(
                    (
                        highest[1].var()
                        +
                        lowest[1].var()
                    )
                    /
                    2
                )

                effect_size = (
                    abs(
                        highest[1].mean()
                        -
                        lowest[1].mean()
                    )
                    /
                    max(
                        pooled_std,
                        1e-9
                    )
                )

                significance = min(
                    1,
                    -np.log10(
                        max(
                            p_value,
                            1e-300
                        )
                    )
                    /
                    10
                )

                score = min(
                    100,
                    effect_size * 40
                    +
                    significance * 50
                )

                if score >= 30:

                    results.append({

                        "Category":
                            category,

                        "Metric":
                            metric,

                        "Higher Group":
                            highest[0],

                        "Lower Group":
                            lowest[0],

                        "Difference":
                            highest[1].mean()
                            -
                            lowest[1].mean(),

                        "Effect Size":
                            round(
                                effect_size,
                                3
                            ),

                        "P-Value":
                            p_value,

                        "Interestingness":
                            round(
                                score,
                                2
                            )
                    })

            except Exception:
                pass

    if not results:

        return pd.DataFrame()

    return (
        pd.DataFrame(results)
        .sort_values(
            "Interestingness",
            ascending=False
        )
        .reset_index(drop=True)
    )


# ================================================================
# ANOMALY DETECTION
# ================================================================

def detect_anomalies(df):

    numeric = df.select_dtypes(
        include=np.number
    ).columns.tolist()

    usable = []

    for column in numeric:

        if df[column].nunique() > 5:

            usable.append(
                column
            )

    if len(usable) < 2:

        return (
            pd.Series(
                False,
                index=df.index
            ),
            []
        )

    X = df[
        usable
    ].copy()

    X = X.replace(
        [np.inf, -np.inf],
        np.nan
    )

    X = X.fillna(
        X.median()
    )

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(
        X
    )

    model = IsolationForest(
        n_estimators=250,
        contamination=.025,
        random_state=42
    )

    prediction = model.fit_predict(
        X_scaled
    )

    mask = (
        prediction == -1
    )

    return (
        pd.Series(
            mask,
            index=df.index
        ),
        usable
    )


# ================================================================
# CLUSTERING
# ================================================================

def discover_clusters(df):

    numeric = df.select_dtypes(
        include=np.number
    ).columns.tolist()

    usable = [
        column
        for column in numeric
        if df[column].nunique() > 5
    ]

    if len(usable) < 3:

        return (
            None,
            None,
            None
        )

    X = df[
        usable
    ].copy()

    X = X.replace(
        [np.inf, -np.inf],
        np.nan
    )

    X = X.fillna(
        X.median()
    )

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(
        X
    )

    best_model = None
    best_score = -1

    for k in range(
        2,
        7
    ):

        model = KMeans(
            n_clusters=k,
            random_state=42,
            n_init=10
        )

        labels = model.fit_predict(
            X_scaled
        )

        if len(
            np.unique(labels)
        ) < 2:

            continue

        score = silhouette_score(
            X_scaled,
            labels
        )

        if score > best_score:

            best_score = score
            best_model = model

    if best_model is None:

        return (
            None,
            None,
            None
        )

    labels = best_model.predict(
        X_scaled
    )

    return (
        labels,
        usable,
        best_score
    )


# ================================================================
# TEMPORAL ANALYSIS
# ================================================================

def temporal_analysis(df):

    if "timestamp" not in df.columns:

        return None

    temp = df.copy()

    temp["timestamp"] = pd.to_datetime(
        temp["timestamp"],
        errors="coerce"
    )

    temp["date"] = (
        temp["timestamp"].dt.date
    )

    daily = (
        temp
        .groupby("date")
        .agg(
            success_rate=(
                "success",
                "mean"
            ),
            average_latency=(
                "latency_ms",
                "mean"
            ),
            average_errors=(
                "errors",
                "mean"
            ),
            events=(
                "user_id",
                "count"
            )
        )
        .reset_index()
    )

    daily["date"] = pd.to_datetime(
        daily["date"]
    )

    daily["rolling_success"] = (
        daily[
            "success_rate"
        ]
        .rolling(
            7,
            min_periods=1
        )
        .mean()
    )

    daily["rolling_latency"] = (
        daily[
            "average_latency"
        ]
        .rolling(
            7,
            min_periods=1
        )
        .mean()
    )

    return daily


# ================================================================
# BUT ENGINE
# ================================================================

def but_engine(df):

    findings = []

    temp = df.copy()

    # ------------------------------------------------------------
    # FRIDAY EXCEPTION
    # ------------------------------------------------------------

    if "timestamp" in temp.columns:

        temp["weekday"] = (
            pd.to_datetime(
                temp["timestamp"],
                errors="coerce"
            ).dt.day_name()
        )

        friday = temp[
            temp["weekday"] == "Friday"
        ]

        other = temp[
            temp["weekday"] != "Friday"
        ]

        if (
            len(friday) > 50
            and
            len(other) > 50
        ):

            difference = (
                other["success"].mean()
                -
                friday["success"].mean()
            )

            score = min(
                100,
                abs(
                    difference
                ) * 300
            )

            if score > 20:

                findings.append({

                    "Type":
                        "Temporal Exception",

                    "Title":
                        "Friday behaves differently",

                    "Observation":
                        f"Friday success rate is "
                        f"{friday['success'].mean()*100:.1f}% "
                        f"versus "
                        f"{other['success'].mean()*100:.1f}% "
                        f"on other days.",

                    "BUT":
                        "The effect may be concentrated in specific actions or environments.",

                    "Evidence":
                        f"Difference = "
                        f"{difference*100:.2f} percentage points.",

                    "Score":
                        score
                })

    # ------------------------------------------------------------
    # MOBILE LATENCY EXCEPTION
    # ------------------------------------------------------------

    if "environment" in temp.columns:

        mobile = temp[
            temp["environment"] == "Mobile"
        ]

        if len(mobile) > 50:

            median_latency = mobile[
                "latency_ms"
            ].median()

            high = mobile[
                mobile["latency_ms"]
                >
                median_latency
            ]

            low = mobile[
                mobile["latency_ms"]
                <=
                median_latency
            ]

            if (
                len(high) > 30
                and
                len(low) > 30
            ):

                difference = (
                    low["success"].mean()
                    -
                    high["success"].mean()
                )

                score = min(
                    100,
                    abs(
                        difference
                    ) * 220
                )

                if score > 20:

                    findings.append({

                        "Type":
                            "Conditional Pattern",

                        "Title":
                            "Latency behaves differently on mobile",

                        "Observation":
                            f"High-latency mobile sessions show a "
                            f"{abs(difference)*100:.1f} "
                            f"percentage-point difference "
                            f"in success.",

                        "BUT":
                            "The same relationship may not exist outside mobile.",

                        "Evidence":
                            f"Mobile sample size = "
                            f"{len(mobile):,}.",

                        "Score":
                            score
                    })

    # ------------------------------------------------------------
    # DURATION SWEET SPOT
    # ------------------------------------------------------------

    if "duration_sec" in temp.columns:

        short = temp[
            temp["duration_sec"] < 50
        ]["success"]

        sweet = temp[
            temp["duration_sec"]
            .between(50, 85)
        ]["success"]

        long = temp[
            temp["duration_sec"] > 85
        ]["success"]

        if min(
            len(short),
            len(sweet),
            len(long)
        ) > 50:

            baseline = max(
                short.mean(),
                long.mean()
            )

            difference = (
                sweet.mean()
                -
                baseline
            )

            score = min(
                100,
                abs(
                    difference
                ) * 280
            )

            findings.append({

                "Type":
                    "Nonlinear Pattern",

                "Title":
                    "A duration sweet spot exists",

                "Observation":
                    f"Sessions between 50–85 seconds show "
                    f"{sweet.mean()*100:.1f}% success.",

                "BUT":
                    "Longer sessions do not necessarily continue improving the outcome.",

                "Evidence":
                    f"Short = {short.mean()*100:.1f}% | "
                    f"Sweet = {sweet.mean()*100:.1f}% | "
                    f"Long = {long.mean()*100:.1f}%",

                "Score":
                    score
            })

    # ------------------------------------------------------------
    # ERROR SEPARATION
    # ------------------------------------------------------------

    low_error = temp[
        temp["errors"] <= 1
    ]["success"]

    high_error = temp[
        temp["errors"] >= 2
    ]["success"]

    if (
        len(low_error) > 50
        and
        len(high_error) > 50
    ):

        difference = (
            low_error.mean()
            -
            high_error.mean()
        )

        score = min(
            100,
            max(
                0,
                difference * 180
            )
        )

        findings.append({

            "Type":
                "Behavioral Pattern",

            "Title":
                "Errors strongly separate outcomes",

            "Observation":
                f"Low-error sessions have "
                f"{low_error.mean()*100:.1f}% success "
                f"versus "
                f"{high_error.mean()*100:.1f}% "
                f"for high-error sessions.",

            "BUT":
                "Errors may be a symptom of difficult sessions rather than the direct cause.",

            "Evidence":
                f"Difference = "
                f"{difference*100:.1f} percentage points.",

            "Score":
                score
        })

    return sorted(
        findings,
        key=lambda x: x["Score"],
        reverse=True
    )


# ================================================================
# DISCOVERY CASE GENERATOR
# ================================================================

def generate_cases(
    relationships,
    groups,
    anomaly_count,
    but_findings
):

    cases = []

    case_id = 1

    # ------------------------------------------------------------
    # RELATIONSHIP CASES
    # ------------------------------------------------------------

    if not relationships.empty:

        for _, row in relationships.head(6).iterrows():

            direction = (
                "positive"
                if row["Pearson"] >= 0
                else
                "negative"
            )

            cases.append({

                "ID":
                    case_id,

                "Title":
                    f"{row['Feature A']} ↔ "
                    f"{row['Feature B']}",

                "Type":
                    "Relationship",

                "Description":
                    f"A {direction} relationship was discovered "
                    f"between {row['Feature A']} and "
                    f"{row['Feature B']}.",

                "Evidence":
                    f"Pearson={row['Pearson']} | "
                    f"Spearman={row['Spearman']} | "
                    f"P={row['P-Value']:.3g}",

                "Score":
                    row["Interestingness"],

                "BUT":
                    "Association does not prove causation."
            })

            case_id += 1

    # ------------------------------------------------------------
    # GROUP CASES
    # ------------------------------------------------------------

    if not groups.empty:

        for _, row in groups.head(5).iterrows():

            cases.append({

                "ID":
                    case_id,

                "Title":
                    f"{row['Category']} separates "
                    f"{row['Metric']}",

                "Type":
                    "Hidden Segment",

                "Description":
                    f"Group {row['Higher Group']} has a higher "
                    f"average {row['Metric']} than "
                    f"group {row['Lower Group']}.",

                "Evidence":
                    f"Effect size={row['Effect Size']:.2f} | "
                    f"P={row['P-Value']:.3g}",

                "Score":
                    row["Interestingness"],

                "BUT":
                    "The difference could be explained by other variables."
            })

            case_id += 1

    # ------------------------------------------------------------
    # ANOMALY CASE
    # ------------------------------------------------------------

    if anomaly_count > 0:

        cases.append({

            "ID":
                case_id,

            "Title":
                f"{anomaly_count:,} unusual records discovered",

            "Type":
                "Anomaly Archaeology",

            "Description":
                "A small population behaves substantially "
                "different from the dominant population.",

            "Evidence":
                f"Isolation Forest identified "
                f"{anomaly_count:,} potential anomalies.",

            "Score":
                min(
                    100,
                    55 +
                    anomaly_count / 20
                ),

            "BUT":
                "Rare does not automatically mean incorrect."
        })

        case_id += 1

    # ------------------------------------------------------------
    # BUT CASES
    # ------------------------------------------------------------

    for finding in but_findings:

        cases.append({

            "ID":
                case_id,

            "Title":
                finding["Title"],

            "Type":
                finding["Type"],

            "Description":
                finding["Observation"],

            "Evidence":
                finding["Evidence"],

            "Score":
                finding["Score"],

            "BUT":
                finding["BUT"]
        })

        case_id += 1

    return sorted(
        cases,
        key=lambda x: x["Score"],
        reverse=True
    )


# ================================================================
# HERO
# ================================================================

st.markdown("""
<div class="hero">

<h1>🔎 AI Shadow Analyst</h1>

<p>
Autonomous Hidden-Pattern Discovery Engine
</p>

<p>
Finds relationships, anomalies, behavioral segments,
temporal shifts and exceptions humans may not think to search for.
</p>

</div>
""", unsafe_allow_html=True)


# ================================================================
# SIDEBAR
# ================================================================

st.sidebar.title(
    "🔬 Investigation Lab"
)

uploaded_file = st.sidebar.file_uploader(
    "Upload your CSV",
    type=["csv"]
)

data_source = st.sidebar.radio(
    "Data Source",
    [
        "Synthetic Investigation Dataset",
        "Uploaded CSV"
    ]
)

investigation_depth = st.sidebar.select_slider(
    "Investigation Depth",
    options=[
        "Quick",
        "Standard",
        "Deep"
    ],
    value="Standard"
)

run_analysis = st.sidebar.button(
    "🔍 START INVESTIGATION",
    use_container_width=True
)

st.sidebar.markdown("---")

st.sidebar.markdown(
    """
### Shadow Analyst searches for

- Hidden correlations
- Unexpected segments
- Statistical differences
- Anomalies
- Behavioral clusters
- Temporal drift
- Nonlinear relationships
- Conditional exceptions
- Outcome drivers
"""
)


# ================================================================
# LOAD DATA
# ================================================================

if (
    data_source == "Uploaded CSV"
    and
    uploaded_file is not None
):

    df = pd.read_csv(
        uploaded_file
    )

    if "timestamp" in df.columns:

        df["timestamp"] = pd.to_datetime(
            df["timestamp"],
            errors="coerce"
        )

else:

    df = generate_data()


# ================================================================
# ANALYSIS
# ================================================================

if (
    run_analysis
    or
    "analysis_results"
    not in st.session_state
):

    progress = st.progress(0)

    status = st.empty()

    status.write(
        "🛰️ Reconnaissance..."
    )

    profile = dataset_profile(
        df
    )

    progress.progress(10)

    status.write(
        "🧬 Mining hidden relationships..."
    )

    relationships = relationship_mining(
        df
    )

    progress.progress(30)

    status.write(
        "🧩 Searching behavioral segments..."
    )

    groups = group_difference_mining(
        df
    )

    progress.progress(50)

    status.write(
        "🕵️ Detecting anomalies..."
    )

    anomaly_mask, anomaly_features = detect_anomalies(
        df
    )

    anomaly_count = int(
        anomaly_mask.sum()
    )

    progress.progress(65)

    status.write(
        "🧠 Discovering behavioral clusters..."
    )

    cluster_labels, cluster_features, silhouette = (
        discover_clusters(
            df
        )
    )

    progress.progress(75)

    status.write(
        "⏳ Investigating time..."
    )

    temporal = temporal_analysis(
        df
    )

    progress.progress(85)

    status.write(
        "⚠️ Running BUT engine..."
    )

    but_findings = but_engine(
        df
    )

    progress.progress(93)

    cases = generate_cases(
        relationships,
        groups,
        anomaly_count,
        but_findings
    )

    progress.progress(100)

    status.success(
        "Investigation complete."
    )

    st.session_state.analysis_results = {

        "profile":
            profile,

        "relationships":
            relationships,

        "groups":
            groups,

        "anomaly_mask":
            anomaly_mask,

        "anomaly_count":
            anomaly_count,

        "cluster_labels":
            cluster_labels,

        "cluster_features":
            cluster_features,

        "silhouette":
            silhouette,

        "temporal":
            temporal,

        "but_findings":
            but_findings,

        "cases":
            cases
    }


# ================================================================
# RETRIEVE RESULTS
# ================================================================

results = (
    st.session_state.analysis_results
)

profile = results["profile"]

relationships = results["relationships"]

groups = results["groups"]

anomaly_mask = results["anomaly_mask"]

anomaly_count = results["anomaly_count"]

cluster_labels = results["cluster_labels"]

cluster_features = results["cluster_features"]

silhouette = results["silhouette"]

temporal = results["temporal"]

but_findings = results["but_findings"]

cases = results["cases"]


# ================================================================
# DATASET INTELLIGENCE
# ================================================================

st.header(
    "📡 Dataset Intelligence"
)

c1, c2, c3, c4, c5 = st.columns(5)

c1.metric(
    "Records",
    f"{profile['rows']:,}"
)

c2.metric(
    "Features",
    profile["columns"]
)

c3.metric(
    "Missing Values",
    f"{profile['missing']:,}"
)

c4.metric(
    "Potential Anomalies",
    f"{anomaly_count:,}"
)

c5.metric(
    "Discoveries",
    len(cases)
)


# ================================================================
# DATA HEALTH
# ================================================================

health_score = 100

health_score -= min(
    30,
    profile["missing_pct"] * 2
)

health_score -= min(
    20,
    profile["duplicate_pct"] * 2
)

health_score = max(
    0,
    min(
        100,
        health_score
    )
)

st.subheader(
    "Dataset Health"
)

h1, h2, h3 = st.columns(3)

h1.metric(
    "Health Score",
    f"{health_score:.1f}/100"
)

h2.metric(
    "Missing %",
    f"{profile['missing_pct']:.2f}%"
)

h3.metric(
    "Duplicate %",
    f"{profile['duplicate_pct']:.2f}%"
)


# ================================================================
# RAW DATA
# ================================================================

with st.expander(
    "📄 Inspect Raw Dataset"
):

    st.dataframe(
        df.head(100),
        use_container_width=True
    )


# ================================================================
# TOP DISCOVERIES
# ================================================================

st.header(
    "🚨 Top Discoveries"
)

if cases:

    for case in cases[:10]:

        if case["Score"] >= 80:

            priority = "HIGH"

        elif case["Score"] >= 60:

            priority = "MEDIUM"

        else:

            priority = "LOW"

        st.markdown(
            f"""
            <div class="card">

            <div class="case-title">
            CASE #{case['ID']} — {case['Title']}
            </div>

            <br>

            <b>Type:</b> {case['Type']}
            &nbsp;&nbsp;&nbsp;

            <b>Priority:</b> {priority}
            &nbsp;&nbsp;&nbsp;

            <b>Interestingness:</b>
            {case['Score']:.1f}/100

            <br><br>

            <b>Discovery</b>

            <br>

            {case['Description']}

            <br><br>

            <b>Evidence</b>

            <br>

            {case['Evidence']}

            <br><br>

            <b>BUT...</b>

            <br>

            {case['BUT']}

            </div>
            """,
            unsafe_allow_html=True
        )


# ================================================================
# DISCOVERY RANKING
# ================================================================

st.header(
    "🏆 Discovery Ranking"
)

if cases:

    ranking = pd.DataFrame({

        "Discovery":
        [
            f"#{case['ID']} {case['Title']}"
            for case in cases[:15]
        ],

        "Score":
        [
            case["Score"]
            for case in cases[:15]
        ],

        "Type":
        [
            case["Type"]
            for case in cases[:15]
        ]
    })

    ranking = ranking.sort_values(
        "Score"
    )

    fig = px.bar(
        ranking,
        x="Score",
        y="Discovery",
        color="Type",
        orientation="h",
        title="Most Interesting Findings"
    )

    fig.update_layout(
        height=650
    )

    st.plotly_chart(
        fig,
        use_container_width=True
    )


# ================================================================
# RELATIONSHIP ARCHAEOLOGY
# ================================================================

st.header(
    "🧬 Relationship Archaeology"
)

if not relationships.empty:

    st.dataframe(
        relationships.head(20),
        use_container_width=True
    )

    strongest = relationships.iloc[0]

    feature_a = strongest[
        "Feature A"
    ]

    feature_b = strongest[
        "Feature B"
    ]

    sample = df.sample(
        min(
            2500,
            len(df)
        ),
        random_state=42
    )

    fig = px.scatter(
        sample,
        x=feature_a,
        y=feature_b,
        trendline="ols",
        opacity=.55,
        title=
        f"{feature_a} vs {feature_b}"
    )

    st.plotly_chart(
        fig,
        use_container_width=True
    )


# ================================================================
# HIDDEN SEGMENTS
# ================================================================

st.header(
    "🧩 Hidden Segment Archaeology"
)

if not groups.empty:

    st.dataframe(
        groups.head(20),
        use_container_width=True
    )

    strongest_group = groups.iloc[0]

    category = strongest_group[
        "Category"
    ]

    metric = strongest_group[
        "Metric"
    ]

    fig = px.box(
        df,
        x=category,
        y=metric,
        color=category,
        title=
        f"{metric} Across {category}"
    )

    st.plotly_chart(
        fig,
        use_container_width=True
    )


# ================================================================
# ANOMALIES
# ================================================================

st.header(
    "🕵️ Anomaly Archaeology"
)

if anomaly_count > 0:

    st.markdown(
        f"""
        <div class="warning">

        <b>{anomaly_count:,}</b>
        observations behave substantially differently
        from the dominant population.

        <br><br>

        An anomaly is unusual — it is not automatically wrong.

        </div>
        """,
        unsafe_allow_html=True
    )

    anomaly_df = df[
        anomaly_mask
    ]

    numeric = df.select_dtypes(
        include=np.number
    ).columns.tolist()

    if len(numeric) >= 2:

        x = numeric[0]

        y = numeric[1]

        normal_df = df[
            ~anomaly_mask
        ].sample(
            min(
                1500,
                (~anomaly_mask).sum()
            ),
            random_state=42
        )

        abnormal_df = anomaly_df.sample(
            min(
                500,
                len(anomaly_df)
            ),
            random_state=42
        )

        normal_df = normal_df.copy()

        abnormal_df = abnormal_df.copy()

        normal_df["Status"] = "Normal"

        abnormal_df["Status"] = "Anomaly"

        combined = pd.concat(
            [
                normal_df,
                abnormal_df
            ]
        )

        fig = px.scatter(
            combined,
            x=x,
            y=y,
            color="Status",
            title="Anomaly Evidence Map",
            opacity=.65
        )

        st.plotly_chart(
            fig,
            use_container_width=True
        )

    with st.expander(
        "View Anomalous Records"
    ):

        st.dataframe(
            anomaly_df.head(100),
            use_container_width=True
        )


# ================================================================
# BEHAVIORAL CLUSTERS
# ================================================================

st.header(
    "🧠 Behavioral Cluster Archaeology"
)

if cluster_labels is not None:

    clustered = df.copy()

    clustered[
        "Behavior Cluster"
    ] = cluster_labels.astype(str)

    c1, c2 = st.columns(2)

    c1.metric(
        "Clusters",
        len(
            np.unique(
                cluster_labels
            )
        )
    )

    c2.metric(
        "Silhouette Score",
        f"{silhouette:.3f}"
    )

    if len(cluster_features) >= 2:

        x = cluster_features[0]

        y = cluster_features[1]

        sample = clustered.sample(
            min(
                2500,
                len(clustered)
            ),
            random_state=42
        )

        fig = px.scatter(
            sample,
            x=x,
            y=y,
            color="Behavior Cluster",
            title="Behavioral Cluster Map",
            opacity=.65
        )

        st.plotly_chart(
            fig,
            use_container_width=True
        )

    cluster_summary = (
        clustered
        .groupby(
            "Behavior Cluster"
        )[cluster_features]
        .mean()
        .round(2)
    )

    st.subheader(
        "Cluster Profiles"
    )

    st.dataframe(
        cluster_summary,
        use_container_width=True
    )


# ================================================================
# TEMPORAL ARCHAEOLOGY
# ================================================================

st.header(
    "⏳ Temporal Archaeology"
)

if temporal is not None:

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=temporal["date"],
            y=temporal["success_rate"],
            mode="lines",
            name="Daily Success"
        )
    )

    fig.add_trace(
        go.Scatter(
            x=temporal["date"],
            y=temporal["rolling_success"],
            mode="lines",
            name="7-Day Rolling Success"
        )
    )

    fig.update_layout(
        title="Behavioral Change Over Time",
        xaxis_title="Date",
        yaxis_title="Success Rate"
    )

    st.plotly_chart(
        fig,
        use_container_width=True
    )

    change = (
        temporal[
            "rolling_success"
        ]
        .diff()
        .abs()
        .fillna(0)
    )

    points = (
        change
        .nlargest(5)
        .index
    )

    st.subheader(
        "Potential Behavioral Shift Points"
    )

    st.dataframe(
        temporal.loc[
            points,
            [
                "date",
                "success_rate",
                "rolling_success",
                "average_latency",
                "average_errors"
            ]
        ],
        use_container_width=True
    )


# ================================================================
# BUT ENGINE
# ================================================================

st.header(
    "⚠️ The BUT Engine"
)

st.markdown(
    """
    <div class="warning">

    <h3>Normal analytics asks: "What pattern exists?"</h3>

    <h3>The BUT Engine asks: "When does that pattern break?"</h3>

    It searches for exceptions, subgroup reversals,
    temporal changes and conditional relationships.

    </div>
    """,
    unsafe_allow_html=True
)

if but_findings:

    for finding in but_findings:

        st.markdown(
            f"""
            <div class="card">

            <div class="case-title">
            {finding['Title']}
            </div>

            <br>

            <b>Observation</b>

            <br>

            {finding['Observation']}

            <br><br>

            <b>Evidence</b>

            <br>

            {finding['Evidence']}

            <br><br>

            <b>BUT...</b>

            <br>

            {finding['BUT']}

            <br><br>

            <b>Investigation Score:</b>
            {finding['Score']:.1f}/100

            </div>
            """,
            unsafe_allow_html=True
        )


# ================================================================
# OUTCOME DRIVER MODEL
# ================================================================

st.header(
    "🎯 Outcome Driver Investigation"
)

if "success" in df.columns:

    numeric = df.select_dtypes(
        include=np.number
    ).columns.tolist()

    features = [
        column
        for column in numeric
        if column != "success"
    ]

    if len(features) >= 2:

        X = df[
            features
        ].copy()

        y = df[
            "success"
        ].copy()

        X = X.replace(
            [np.inf, -np.inf],
            np.nan
        )

        X = X.fillna(
            X.median()
        )

        model = RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            max_depth=8
        )

        model.fit(
            X,
            y
        )

        importance = pd.DataFrame({

            "Feature":
                features,

            "Importance":
                model.feature_importances_

        }).sort_values(
            "Importance",
            ascending=False
        )

        fig = px.bar(
            importance.head(12).sort_values(
                "Importance"
            ),
            x="Importance",
            y="Feature",
            orientation="h",
            title="Most Informative Variables"
        )

        st.plotly_chart(
            fig,
            use_container_width=True
        )

        st.info(
            "Feature importance indicates predictive usefulness, "
            "not causation."
        )


# ================================================================
# CASE FILES
# ================================================================

st.header(
    "📁 Investigation Case Files"
)

for case in cases:

    with st.expander(
        f"CASE #{case['ID']} | "
        f"{case['Title']} | "
        f"{case['Score']:.1f}/100"
    ):

        st.write(
            "**Type:**",
            case["Type"]
        )

        st.write(
            "**Discovery:**",
            case["Description"]
        )

        st.write(
            "**Evidence:**",
            case["Evidence"]
        )

        st.write(
            "**BUT:**",
            case["BUT"]
        )

        st.progress(
            int(
                min(
                    100,
                    case["Score"]
                )
            )
        )


# ================================================================
# REPORT GENERATION
# ================================================================

st.header(
    "📄 Investigation Report"
)

if cases:

    strongest = cases[0]

    report = f"""
# AI SHADOW ANALYST

## Autonomous Hidden-Pattern Discovery Report

---

## Dataset Overview

Records: {profile['rows']:,}

Features: {profile['columns']}

Missing values: {profile['missing']:,}

Duplicates: {profile['duplicates']:,}

Potential anomalies: {anomaly_count:,}

Dataset health: {health_score:.1f}/100

Total discoveries: {len(cases)}

---

## Strongest Discovery

Case #{strongest['ID']}

Title:
{strongest['Title']}

Type:
{strongest['Type']}

Interestingness:
{strongest['Score']:.1f}/100

Discovery:
{strongest['Description']}

Evidence:
{strongest['Evidence']}

BUT:
{strongest['BUT']}

---

## Discovery List

"""

    for number, case in enumerate(
        cases[:15],
        1
    ):

        report += f"""
### {number}. {case['Title']}

Type:
{case['Type']}

Score:
{case['Score']:.1f}/100

Discovery:
{case['Description']}

Evidence:
{case['Evidence']}

BUT:
{case['BUT']}

---

"""

    report += """

## Interpretation

The system identifies statistical associations,
unusual observations, behavioral segments and
potential temporal changes.

These discoveries should not automatically be interpreted
as causal relationships.

Controlled experiments and domain knowledge are required
for causal conclusions.

## Recommended Next Step

Prioritize the strongest discovery for deeper investigation.

Generated by AI Shadow Analyst.
"""

    st.download_button(
        "⬇️ Download Full Investigation Report",
        data=report,
        file_name="AI_Shadow_Analyst_Report.md",
        mime="text/markdown"
    )


# ================================================================
# COMPLETE DISCOVERY DATABASE
# ================================================================

st.header(
    "🗂️ Complete Discovery Database"
)

if cases:

    discovery_table = pd.DataFrame(
        cases
    )

    st.dataframe(
        discovery_table,
        use_container_width=True,
        hide_index=True
    )


# ================================================================
# FOOTER
# ================================================================

st.markdown(
    """
    <hr>

    <div style="text-align:center">

    <h2>🔎 AI Shadow Analyst</h2>

    <p>
    Finding the questions hidden inside your data.
    </p>

    <p>
    Python • Pandas • NumPy • SciPy • Scikit-learn • Plotly • Streamlit
    </p>

    </div>
    """,
    unsafe_allow_html=True
)
'''


# ================================================================
# WRITE STREAMLIT FILE
# ================================================================

app_path = "/content/ai_shadow_analyst.py"

with open(
    app_path,
    "w",
    encoding="utf-8"
) as file:

    file.write(app_code)


# ================================================================
# STOP OLD STREAMLIT PROCESSES
# ================================================================

os.system(
    "pkill -f 'streamlit run' 2>/dev/null || true"
)

time.sleep(2)


# ================================================================
# START STREAMLIT
# ================================================================

process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        app_path,
        "--server.port",
        "8501",
        "--server.address",
        "0.0.0.0",
        "--server.headless",
        "true",
        "--browser.gatherUsageStats",
        "false"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)


# ================================================================
# WAIT
# ================================================================

time.sleep(5)


# ================================================================
# GOOGLE COLAB URL
# ================================================================

try:

    from google.colab.output import eval_js

    app_url = eval_js(
        "google.colab.kernel.proxyPort(8501)"
    )

    print()
    print("=" * 80)
    print("                 AI SHADOW ANALYST")
    print("=" * 80)
    print()
    print("Application started successfully.")
    print()
    print("OPEN THIS URL:")
    print(app_url)
    print()
    print("=" * 80)

except Exception:

    print()
    print("=" * 80)
    print("AI SHADOW ANALYST STARTED")
    print("=" * 80)
    print()
    print("Open: http://localhost:8501")
    print()
    print("=" * 80)


                 AI SHADOW ANALYST

Application started successfully.

OPEN THIS URL:
https://8501-m-s-kkb-usw1c1-21s3yb0ghnqsz-c.us-west1-1.prod.colab.dev

